# 毫米波雷达数据处理教学（FMCW 距离-多普勒-角度 → 点云）

本 Notebook 逐步讲解**从仿真导出的二进制原始数据到点云**的完整处理链，每一步都先解释公式、再画图验证。

数据来自 **2D 场景雷达仿真器**（`radar-sim-cli` 或 GUI 的「导出数据」），目录里包含：

| 文件 | 内容 |
| --- | --- |
| `signal.bin` | 无文件头的 int16 原始 IQ（布局由 `meta.json` 描述） |
| `meta.json` | 雷达参数、派生指标、场景快照、车辆轨迹、帧索引 |
| `pointcloud_gt.csv` | 理想点云真值（用于对比） |

> 关键：**所有参数都从 `meta.json` 读**，不要硬编码。处理公式见 `docs/DESIGN.md` 第 10.6 节「防坑清单」。

## 1. 加载数据，先看参数

`RadarFile` 会读取 `meta.json` 并据此解析 `signal.bin`（同时支持 TI DCA1000 原生布局与简化交错 IQ 布局）。

**注意**：数据是**复数 IQ**，不是实信号。复采样使差拍频率的符号可知，因此最大不模糊距离是 $c f_s/(2S)$，`Ns` 个距离 bin 全部可用。

In [ ]:
import pathlib, sys

# 让 Notebook 能找到 templates/python 下的 radar_io.py / cfar.py
for base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (base / 'templates' / 'python' / 'radar_io.py').exists():
        sys.path.insert(0, str(base / 'templates' / 'python')); PROJECT = base; break
    if (base / 'radar_io.py').exists():
        sys.path.insert(0, str(base)); PROJECT = base.parent.parent; break
else:
    raise RuntimeError('找不到 radar_io.py')

import numpy as np
import matplotlib.pyplot as plt

matplotlib_rc = plt.rcParams
matplotlib_rc['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans']
matplotlib_rc['axes.unicode_minus'] = False

from radar_io import RadarFile, periodic_window
from cfar import extract_pointcloud, ca_cfar_2d, ca_cfar_alpha, pick_peaks

DATASET = PROJECT / 'dataset' / 'demo'   # ← 改成你自己的导出目录
rf = RadarFile(DATASET)
print(rf.summary())

## 2. 原始 IQ 与距离像

单散射点在差拍后是：

$$x[n] = A\exp\!\left(j2\pi \frac{f_b}{f_s} n\right),\qquad f_b = \frac{2SR}{c}$$

对快时间做 FFT，峰值所在 bin $k$ 对应距离

$$R[k] = k\cdot\frac{c f_s}{2 S N_s} = k\cdot\Delta R,\qquad \Delta R = \frac{c}{2B_{valid}},\ B_{valid}=S\frac{N_s}{f_s}$$

⚠️ 常见错误：用整段扫频带宽 $B = S T_r$ 算分辨率——真正的分辨率由**采样窗口内**扫过的带宽决定。

In [ ]:
cube = rf.frame(0)
print('cube shape (Nc, Nv, Ns) =', cube.shape, cube.dtype)
print(f'ΔR = {rf.range_bin_m:.4f} m,  Rmax = {rf.max_range_m:.2f} m')

w = periodic_window(rf.n_samples, 'hann')[None, None, :]
profile = np.fft.fft(cube * w, axis=-1)
profile_db = 20 * np.log10(np.maximum(np.sqrt((np.abs(profile) ** 2).sum(axis=(0, 1))), 1e-30))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
axes[0].plot(cube[0, 0, :64].real, label='I'); axes[0].plot(cube[0, 0, :64].imag, label='Q')
axes[0].set_title('原始 IQ（chirp0 / 阵元0）'); axes[0].set_xlabel('采样点'); axes[0].legend(); axes[0].grid(alpha=.25)
axes[1].plot(rf.range_axis(), profile_db)
axes[1].set_title('距离像'); axes[1].set_xlabel('距离 (m)'); axes[1].set_ylabel('dB')
axes[1].set_xlim(0, rf.max_range_m); axes[1].grid(alpha=.25)
plt.tight_layout()

k = int(np.argmax(profile_db))
print(f'最强距离峰: bin {k} → {rf.range_axis()[k]:.2f} m')

## 3. 距离-多普勒图

目标径向速度 $v$ 会让慢时间（chirp 间）相位线性变化：

$$x[m] \propto \exp\!\left(j2\pi \frac{2v}{\lambda} m T_c\right)$$

（约定：**正径向速度 = 远离**，这由回波相位 $\varphi = 4\pi R/\lambda$ 推出。）

$$
v_{\max} = \pm\frac{\lambda}{4T_c},\qquad \Delta v = \frac{\lambda}{2N_c T_c}
$$

**⚠️ 千万不要用 `linspace(-vmax, vmax, Nc)` 构造速度轴**：步长会错，而且 $N_c$ 为偶数时轴里根本没有 0 速 bin。正确写法是 $(i - N_c/2)\Delta v$。

In [ ]:
rd_db, r_axis, v_axis = rf.range_doppler(0)
print(f'vmax = ±{rf.max_velocity_mps:.3f} m/s,  Δv = {rf.velocity_resolution_mps:.4f} m/s')
print('零速 bin 索引 =', rf.n_chirps // 2, '→ v =', v_axis[rf.n_chirps // 2])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
im = axes[0].imshow(rd_db, aspect='auto', origin='lower', cmap='turbo',
                    extent=(0, rf.max_range_m, -rf.max_velocity_mps, rf.max_velocity_mps),
                    vmin=rd_db.max() - 45, vmax=rd_db.max())
plt.colorbar(im, ax=axes[0], label='dB')
axes[0].set_title('距离-多普勒'); axes[0].set_xlabel('距离 (m)'); axes[0].set_ylabel('速度 (m/s)')

axes[1].plot(v_axis, rd_db[:, :20].max(axis=1))
axes[1].set_title('速度剖面（前 20 个距离 bin 的最大值）'); axes[1].set_xlabel('速度 (m/s)'); axes[1].grid(alpha=.25)
plt.tight_layout()

## 4. 角度测量（阵列 FFT）

$N_v$ 个间距 $d$ 的虚拟阵元，来自方位 $\theta$ 的回波有线性相位斜坡

$$x[k] \propto \exp\!\left(j2\pi \frac{d\,k\sin\theta}{\lambda}\right)$$

对空间维做 FFT，bin $k$ 对应

$$\theta[k] = \arcsin\!\left(\frac{(k - N_{fft}/2)\,\lambda}{N_{fft}\, d}\right)$$

⚠️ 两个坑：
1. **不能用线性角度近似** $\theta \approx (k-N/2)\cdot 180/N$，大角度会严重失真；
2. **快拍必须取自复数距离-多普勒立方体**（已完成相干积累），只对几个 chirp 相加会丢掉约 20 dB 增益。

In [ ]:
rd_cube = rf.range_doppler_complex(0)
i0, k0 = np.unravel_index(int(np.argmax(rd_db)), rd_db.shape)
print(f'最强单元: 距离 {r_axis[k0]:.2f} m, 速度 {v_axis[i0]:+.3f} m/s')

n_fft = 180
snap = rd_cube[i0, :, k0] * periodic_window(rf.n_elements, 'hann')
spectrum = np.abs(np.fft.fftshift(np.fft.fft(snap, n_fft))) ** 2
angle = np.rad2deg(rf.angle_axis(n_fft))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
axes[0].stem(np.rad2deg(np.angle(snap)), basefmt=' ')
axes[0].set_title('阵列快拍相位（线性斜坡 → 有方向）'); axes[0].set_xlabel('相位 (°)'); axes[0].grid(alpha=.25)
axes[1].plot(angle, 10 * np.log10(np.maximum(spectrum, 1e-30)))
axes[1].set_title('空间谱'); axes[1].set_xlabel('方位 (°)'); axes[1].set_ylabel('dB')
axes[1].axvline(angle[int(np.argmax(spectrum))], color='r', ls='--')
axes[1].grid(alpha=.25)
plt.tight_layout()
print(f'测得方位 = {angle[int(np.argmax(spectrum))]:.2f}°，角分辨率 ≈ {np.degrees(2 / rf.n_elements):.2f}°')

## 5. CFAR 检测与点云

CA-CFAR 用保护窗外的参考单元估杂波功率，阈值

$$T = \alpha\,\bar P_{noise},\qquad \alpha = N_{ref}\left(P_{fa}^{-1/N_{ref}} - 1\right)$$

这样给定的虚警概率 $P_{fa}$ 才有意义。常见的 `mean + 1.5σ` 写法无法控制虚警率。

In [ ]:
pd = 10 ** (rd_db / 10)
mask, threshold, noise_mean, alpha = ca_cfar_2d(pd, pfa=1e-3)
di, ki = pick_peaks(pd, mask, max_peaks=64)
print(f'alpha = {alpha:.3f},  CFAR 命中数 = {mask.sum()},  非极大抑制后 = {di.size}')

cloud = extract_pointcloud(rf, 0, pfa=1e-3)
print(f"\n{'距离(m)':>9}{'速度(m/s)':>12}{'方位(°)':>10}{'SNR(dB)':>10}")
for i in np.argsort(-cloud['power_db'])[:10]:
    print(f"{cloud['range_m'][i]:9.2f}{cloud['velocity_mps'][i]:+12.3f}"
          f"{cloud['azimuth_deg'][i]:10.2f}{cloud['snr_db'][i]:10.1f}")

## 6. 与真值对比

`pointcloud_gt.csv` 里有每个散射点/多径路径的理想 $(R, \theta, v, \text{幅度})$。

注意：**CFAR 检测的是分辨单元，不是单个散射点**。一面被离散成 100 段的墙只会产生几个检测，所以不要用「散射点逐点命中率」评价算法——要看**精度**（RMS 误差）与**虚警率**。

完整统计脚本见 `templates/python/compare_gt.py`。

In [ ]:
truth = rf.truth(0)
direct = np.asarray(truth['path_type']) == 'direct'

fig, ax = plt.subplots(figsize=(6, 4.6))
if cloud['x'].size:
    sc = ax.scatter(cloud['x'], cloud['y'], c=cloud['snr_db'], cmap='autumn', s=30, label='CFAR 检测')
    plt.colorbar(sc, ax=ax, label='SNR (dB)')
r = np.asarray(truth['range_m'])[direct]; a = np.deg2rad(np.asarray(truth['azimuth_deg'])[direct])
ax.scatter(r * np.cos(a), r * np.sin(a), marker='+', c='lime', s=30, label='真值（直接路径）')
ax.plot([0], [0], marker='^', ms=10, color='cyan', label='雷达')
ax.set_aspect('equal', adjustable='datalim'); ax.grid(alpha=.25); ax.legend()
ax.set_title('点云 vs 真值（雷达坐标系）'); ax.set_xlabel('前方 X (m)'); ax.set_ylabel('横向 Y (m)')
plt.tight_layout()

## 7. 参数实验

改雷达参数重新导出，观察分辨率与数据量的变化：

```bash
# 更宽的带宽 → 更好的距离分辨率（数据量不变）
radar-sim-cli --demo --out dataset/wide --chirps 128 --samples 512

# TDM-MIMO（硬件保真）：速度范围降为 ±2 m/s，但速度分辨率不变
radar-sim-cli --demo --out dataset/tdm --mimo tdm --chirps 126
```

对比要点：

| 改什么 | $\Delta R$ | $R_{max}$ | $v_{max}$ | $\Delta v$ | 单帧大小 |
| --- | --- | --- | --- | --- | --- |
| $N_s$ ↑ | ↓（分母变大） | ↑ | — | — | ↑ |
| $N_c$ ↑ | — | — | — | ↓ | ↑ |
| $N_t$ ↑（tdm） | — | — | ↓ | — | ↑ |

最后别忘了关掉文件句柄：

In [ ]:
rf.close()